In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
ds = pd.read_csv('books.csv')

In [3]:
ds.shape

(10000, 23)

In [4]:
ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         10000 non-null  int64  
 1   book_id                    10000 non-null  int64  
 2   best_book_id               10000 non-null  int64  
 3   work_id                    10000 non-null  int64  
 4   books_count                10000 non-null  int64  
 5   isbn                       9300 non-null   object 
 6   isbn13                     9415 non-null   float64
 7   authors                    10000 non-null  object 
 8   original_publication_year  9979 non-null   float64
 9   original_title             9415 non-null   object 
 10  title                      10000 non-null  object 
 11  language_code              8916 non-null   object 
 12  average_rating             10000 non-null  float64
 13  ratings_count              10000 non-null  int6

In [5]:
ds.head()

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [6]:
# Since this project is about predicting a book's average rating (average_rating), we are starting by dropping ratings_1 - ratings_5, as well as work_ratings_count (their sum),
# because they are used directly in the formula for average_rating and would leak the target (data leakage).

# ratings_count and work_text_reviews_count, on the other hand, are separate numbers that are not directly part of that formula - they reflect reader popularity/engagement, not the rating itself.
# For that reason, we are keeping them as features.

# We are also dropping the image columns (image_url and small_image_url) and all the ids and isbns, since they have no predictive value for average_rating.

# Keeping books_count, since it represents the number of editions of a book and could reasonably correlate with popularity.

In [7]:
ds = ds.drop(columns=['ratings_1', 'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'work_ratings_count', 'image_url', 'small_image_url', 'id', 'book_id', 'best_book_id', 'work_id', 'isbn', 'isbn13'])
ds.head()

,books_count,authors,original_publication_year,original_title,title,language_code,average_rating,ratings_count,work_text_reviews_count
0,272,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",eng,4.34,4780653,155254
1,491,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44,4602479,75867
2,226,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",en-US,3.57,3866839,95009
3,487,Harper Lee,1960.0,To Kill a Mockingbird,To Kill a Mockingbird,eng,4.25,3198671,72586
4,1356,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,eng,3.89,2683664,51992


In [8]:
# Since most models cannot use raw text directly, we are going to drop the title columns.
# But before that, we will use them to create a few new columns.

In [9]:
# First we are making columns with information about title and original title lenghts.

ds['title_length'] = ds['title'].str.len()
ds['original_title_length'] = ds['original_title'].str.len()

In [10]:
# We are making a column that checks if the title contains '#' (usually followed by a number, like "#1"), which means the book is likely part of a series.
# This is not perfect, because some series books might not have '#' in the title, so this only catches what the title format shows us, not the full truth.

ds['series'] = ds['title'].str.contains('#')

In [11]:
ds = ds.drop(columns=['title', 'original_title'])

In [12]:
ds.head()

,books_count,authors,original_publication_year,language_code,average_rating,ratings_count,work_text_reviews_count,title_length,original_title_length,series
0,272,Suzanne Collins,2008.0,eng,4.34,4780653,155254,39,16.0,True
1,491,"J.K. Rowling, Mary GrandPré",1997.0,eng,4.44,4602479,75867,56,40.0,True
2,226,Stephenie Meyer,2005.0,en-US,3.57,3866839,95009,23,8.0,True
3,487,Harper Lee,1960.0,eng,4.25,3198671,72586,21,21.0,False
4,1356,F. Scott Fitzgerald,1925.0,eng,3.89,2683664,51992,16,16.0,False


In [13]:
# Now, we will handle missing values.

ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   books_count                10000 non-null  int64  
 1   authors                    10000 non-null  object 
 2   original_publication_year  9979 non-null   float64
 3   language_code              8916 non-null   object 
 4   average_rating             10000 non-null  float64
 5   ratings_count              10000 non-null  int64  
 6   work_text_reviews_count    10000 non-null  int64  
 7   title_length               10000 non-null  int64  
 8   original_title_length      9415 non-null   float64
 9   series                     10000 non-null  bool   
dtypes: bool(1), float64(3), int64(4), object(2)
memory usage: 713.0+ KB


In [14]:
# We see that only original_publication_year, language_code, and original_title_length have missing values.
# original_publication_year has very few missing values (21 out of 10,000), so we are dropping those rows.

# original_title_length comes from the original_title column, which we already dropped. There's no good way to guess the length of text we don't have, so we're dropping these rows too - even 
# though the percentage (just over 5%) is a bit higher than what we'd normally fill instead of drop.

ds = ds.loc[ds[['original_publication_year', 'original_title_length']].notna().all(axis=1)]
ds.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9409 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   books_count                9409 non-null   int64  
 1   authors                    9409 non-null   object 
 2   original_publication_year  9409 non-null   float64
 3   language_code              8405 non-null   object 
 4   average_rating             9409 non-null   float64
 5   ratings_count              9409 non-null   int64  
 6   work_text_reviews_count    9409 non-null   int64  
 7   title_length               9409 non-null   int64  
 8   original_title_length      9409 non-null   float64
 9   series                     9409 non-null   bool   
dtypes: bool(1), float64(3), int64(4), object(2)
memory usage: 744.3+ KB


In [15]:
print(ds['language_code'].unique())

['eng' 'en-US' 'en-CA' nan 'spa' 'en-GB' 'fre' 'nl' 'ara' 'por' 'ger'
 'nor' 'jpn' 'en' 'vie' 'ind' 'pol' 'tur' 'dan' 'fil' 'ita' 'per' 'swe'
 'rum' 'rus']


In [16]:
# English appears under several different codes. We will group eng, en-US, en-CA, en-GB, and en together under a single code - eng.

# For the missing (NaN) values, we won't assume they are English. It's more likely these are less common languages that just weren't recorded. 
# We are putting them in a new category instead - other.

variants = ['en-US', 'en-CA', 'en-GB', 'en']
ds.loc[ds['language_code'].isin(variants), 'language_code'] = 'eng'

ds['language_code'] = ds['language_code'].fillna('other')

print(ds['language_code'].unique())

['eng' 'other' 'spa' 'fre' 'nl' 'ara' 'por' 'ger' 'nor' 'jpn' 'vie' 'ind'
 'pol' 'tur' 'dan' 'fil' 'ita' 'per' 'swe' 'rum' 'rus']


In [17]:
print(ds['language_code'].isna().sum())

0


In [18]:
# author_books_count represents the total number of books associated with each author in the dataset.
# It is created before the train/test split because it does not use the target variable.

ds['author_books_count'] = ds['authors'].map(ds['authors'].value_counts())

In [19]:
# Now that missing values are handled, we will move on to outliers.

ds.describe()

,books_count,original_publication_year,average_rating,ratings_count,work_text_reviews_count,title_length,original_title_length,author_books_count
count,9409.000000,9409.000000,9409.000000,9.409000e+03,9409.000000,9409.000000,9409.000000,9409.000000
mean,78.047295,1981.476034,3.998695,5.567878e+04,2989.861622,31.958231,22.991072,7.488043
std,170.831533,152.682468,0.252048,1.616107e+05,6236.436306,18.969320,18.036867,10.190119
min,1.000000,-1750.000000,2.470000,2.716000e+03,3.000000,2.000000,1.000000,1.000000
25%,25.000000,1989.000000,3.850000,1.373000e+04,701.000000,18.000000,12.000000,1.000000
50%,42.000000,2004.000000,4.010000,2.164600e+04,1412.000000,30.000000,17.000000,3.000000
75%,69.000000,2010.000000,4.170000,4.242600e+04,2845.000000,41.000000,27.000000,10.000000
max,3455.000000,2017.000000,4.820000,4.780653e+06,155254.000000,186.000000,196.000000,59.000000


In [20]:
print((ds['original_title_length'] == 1).sum())

6


In [21]:
# books_count, average_rating, ratings_count and work_text_reviews_count all look realistic.
# original_publication_year seemed to have a problem because of negative values, but books written before the common era (e.g. The Iliad, The Odyssey) have negative years, so these 
# values are correct and we are keeping them.
# title_length has a minimum of 2, but short titles do exist (e.g. Stephen King's famous horror "It"), so this is realistic too.
# original_title_length has a minimum of 1, which turned out to be rows where original_title was just a single space (" "), not a real title. We are treating this as a missing value and dropping 
# these rows (only 6 rows), the same way we handled the actual NaN values in this column.

In [22]:
ds = ds.loc[ds['original_title_length']>1]
print((ds['original_title_length'] == 1).sum())

0


In [23]:
ds.describe()

,books_count,original_publication_year,average_rating,ratings_count,work_text_reviews_count,title_length,original_title_length,author_books_count
count,9403.000000,9403.000000,9403.000000,9.403000e+03,9403.000000,9403.000000,9403.000000,9403.000000
mean,78.083484,1981.464001,3.998687,5.570398e+04,2991.334042,31.953525,23.005105,7.486760
std,170.879962,152.730307,0.252101,1.616591e+05,6238.111310,18.968858,18.034061,10.190648
min,1.000000,-1750.000000,2.470000,2.716000e+03,3.000000,2.000000,2.000000,1.000000
25%,25.000000,1989.000000,3.850000,1.373150e+04,702.000000,18.000000,12.000000,1.000000
50%,42.000000,2004.000000,4.010000,2.166200e+04,1413.000000,30.000000,18.000000,3.000000
75%,69.000000,2010.000000,4.170000,4.243600e+04,2845.500000,41.000000,27.000000,10.000000
max,3455.000000,2017.000000,4.820000,4.780653e+06,155254.000000,186.000000,196.000000,59.000000


In [24]:
# Unlike other similar projects, none of the extreme values here appear to be data errors - they are all explainable.
# Since IQR filtering would likely remove legitimate and interesting data points rather than actual errors, we are not applying it here.

In [25]:
print(ds.nunique())
print(ds.dtypes)

books_count                   590
authors                      4410
original_publication_year     285
language_code                  21
average_rating                182
ratings_count                8542
work_text_reviews_count      4489
title_length                  136
original_title_length         133
series                          2
author_books_count             36
dtype: int64
books_count                    int64
authors                       object
original_publication_year    float64
language_code                 object
average_rating               float64
ratings_count                  int64
work_text_reviews_count        int64
title_length                   int64
original_title_length        float64
series                          bool
author_books_count             int64
dtype: object


In [ ]:
# For column authors we are using target encoding since it has 4410 values.
# Column language_code has smaller number of unique values, so we are going to use OHE for encoding it.

In [27]:
# Before target encoding (when we are grouping by target) we must do train/test split first. This way we are avoiding data leakage.
X = ds.drop(columns=['average_rating'])
y = ds['average_rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
author_rating_map = X_train.join(y_train).groupby('authors')['average_rating'].mean()

In [29]:
X_train['authors'] = X_train['authors'].map(author_rating_map)
X_test['authors'] = X_test['authors'].map(author_rating_map)

In [30]:
# Potential problem here - if an author exists in the test set but not in the train set, map() will return NaN.

print(X_test['authors'].isna().sum())

666


In [31]:
unseen_authors = X_test['authors'].isna().sum()
print(unseen_authors / len(X_test) * 100)

35.406698564593306


In [32]:
# Around 35% of the authors in the test set were not in the training set. This is a limitation of the dataset (many authors appear only once or twice).
# For unseen authors we are using the global mean rating from the training set.

global_mean_rating = y_train.mean()
X_test['authors'] = X_test['authors'].fillna(global_mean_rating)

In [33]:
ds.head()

,books_count,authors,original_publication_year,language_code,average_rating,ratings_count,work_text_reviews_count,title_length,original_title_length,series,author_books_count
0,272,Suzanne Collins,2008.0,eng,4.34,4780653,155254,39,16.0,True,9
1,491,"J.K. Rowling, Mary GrandPré",1997.0,eng,4.44,4602479,75867,56,40.0,True,7
2,226,Stephenie Meyer,2005.0,eng,3.57,3866839,95009,23,8.0,True,10
3,487,Harper Lee,1960.0,eng,4.25,3198671,72586,21,21.0,False,2
4,1356,F. Scott Fitzgerald,1925.0,eng,3.89,2683664,51992,16,16.0,False,5


In [34]:
# The data is ready for training and evaluating the machine learning models.